# Assessment Cover Page


**Student Full Name**   Vanessa Costa de Brito

**Student Number** 	    2025416

**Module Title**        Machine Learning - FT

**Assessment Title**    CA  1 - Bike-Sharing Usage Patterns and Demand Forecasting

**Lecturer/Supervisor**	Dr. Muhammad Iqbal

**Assessment Due Date**	9th September 2026

**Date of Submission** 	30th October 2026

**Dataset**             day.csv


## Use of AI Tools
**I acknowledge the use of ChatGPT to improve grammar, brainstorming.**


<div style="width: 700px; text-align: justify; text-justify: inter-word;">

<p>This notebook covers data preparation, exploratory analysis, a comparison of two clustering algorithms, and the evaluation of ARIMA and SARIMA forecasts.</p>

</div>

   
## 1. Introduction and Objectives

<div style="width: 700px; text-align: justify; text-justify: inter-word;">
    

### 1.1 - Introduction
<p>Bike-sharing systems are a form of urban transportation. The number of rentals can vary over time, depending on factors such as adverse conditions and the specific day of the week. This project analyzes these patterns using historical data from Capital Bikeshare, provided by the UCI (Fanaee-T, 2013).</p>

### 1.2 - Objectives
<p>The chosen topic allows for the application of machine learning techniques to an everyday scenario, with results that can be interpreted without the need for a complex business context. The objective is to investigate how days can be grouped based on their characteristics and to what extent it is possible to predict the total number of daily rentals.</p>

### 1.3 - The Analytical Questions Are: 

 
* <p>(1) What differences emerge between days, including the impact of adverse conditions?; </p>
  
* <p>(2) Which clustering solution offers the best balance between distinct separation and interpretability? ; and </p> 
    
* <p>(3) How do ARIMA and SARIMA models compare to a simple seasonal forecast? </p>


<p>This study does not aim to establish causality or estimate current demand. Each observation represents a single day, therefore, the clusters reflect profiles of days rather than individual customer profiles. Clustering is an unsupervised learning task.</p>

<p>Time-series forecasting constitutes the second task addressed in this analysis.</p>


</div>

## 02. Data Source and Data Peparation

<div style="width: 700px; text-align: justify; text-justify: inter-word;">


<p>The dataset is available at the following link as mentioned below:</p>

##### <p>https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset<p>
<p>This uses the original daily file provided for the project, without generating any synthetic data. The date serves as a temporal index, <code>instant</code> is simply an identifier, and <code>cnt</code> corresponds to the sum of <code>casual</code> and <code>registred</code> users. Temperature, feels-like temperature, humidity, and wind speed are normalized in the original dataset. </p>
   
<p>Categorical variables encoded as numbers do not automatically represent continuous scales. </p>

</div>

In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn, statsmodels

import sys, platform, warnings

from pathlib import Path

from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning

In [5]:
random_state=42

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': .2})

df = pd.read_csv('day.csv')


display(df.head())

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


## 03. Data Dictionary and Data Quality

<div style="width: 700px; text-align: justify; text-justify: inter-word;">
    
Variable Name |	Role |	Type | Description	Units |	Missing Values
|:---|:---|:---|:---:|:---:|
instant | ID | Integer | record index| no
dteday | Feature | Date | date | no
season | Feature | Categorical | 1:winter, 2:spring, 3:summer, 4:fall | no
yr | Feature | Categorical | year (0:2011, 1:2012) | no
mnth | Feature | Categorical | month (1 to 12) | no
hr | Feature | Categorical | hour (0 to 23) | no
holiday | Feature | Binary | weather day is holiday or not (extracted from http://dchr.dc.gov/page/holiday-schedule) | no
weekday | Feature | Categorical | day of the week | no
workingday | Feature |Binary | if day is neither weekend or holiday is 1, otherwise is 0 | no
weathersit | Feature | Categorical|  - 1: Clear, Few clouds, Partly cloudy, Partly cloudy | no
temp | Feature | Continuous | Normalized temperature in Celsius. The values ​​are derived via (t-t_min)/(t_max-t_min), t_min=-8, t_max=+39 (only in hourly scale) C | no
atemp | Feature | Continuous | Normalized feeling temperature in Celsius. The values ​​are derived via (t-t_min)/(t_max-t_min), t_min=-16, t_max=+50 (only in hourly scale) C | no
hum | Feature | Continuous | Normalized humidity. The values are divided to 100 (max) | no
windspeed | Feature | Continuous | Normalized wind on speed. The values are divided to 67 (max) | no
casual | Other | Integer | count of casual users | no
registered | Other | Integer | count of registered users | no
cnt | Target | Integer | count of total rental bikes including both casual and registered | no







<p>Cleanup is guided by verification, rather than by a need to eliminate records. Days with very low demand may represent actual observations and will be preserved, unless there is evidence of an error.</p>

</div>

In [6]:
# Save a copy of the original data
df_original = df.copy()

#Converting the column to date format
df['dteday'] = pd.to_datetime(df['dteday'], errors='raise')

# Order of dates and indices
df = df.sort_values('dteday').reset_index(drop=True)

# Create the expected calendar
expected = pd.date_range(df['dteday'].min(), df['dteday'].max(), freq='D')


In [8]:
# Check data quality
checks = pd.Series({'lines': len(df),'columns': df.shape[1], 'missing cells': int(df.isna().sum().sum()),
    'duplicate line': int(df.duplicated().sum()), 'duplicate data': int(df['dteday'].duplicated().sum()),
    'missing days calendar': len(expected.difference(df['dteday']) ),'erros casual + registered = cnt': int(
        (df['casual'] + df['registered'] != df['cnt']).sum()),'negative count':
        int((df[['casual', 'registered', 'cnt']] < 0).sum().sum())})

display(checks.to_frame('Result'))
display(df.describe().T)

# Check expected conditions
assert df.isna().sum().sum() == 0
assert not df['dteday'].duplicated().any()
assert len(expected.difference(df['dteday'])) == 0
assert (df['casual'] + df['registered'] == df['cnt']).all()
assert (df[['casual', 'registered', 'cnt']] >= 0).all().all()

# Check the normalized variables
for col in ['temp', 'atemp', 'hum', 'windspeed']:
    assert df[col].between(0, 1).all(), f'Check Interval of {col}'

# Remove the identifier from the prepared base
clean = df.drop(columns='instant').copy()

# Confirm the minimum number of columns required in CA1
assert clean.shape[1] >= 7

print(f'Data Preparation: {clean.shape[0]} Days and '
    f'{clean.shape[1]} colunms. No lines were removed.')

,Result
lines,731
columns,16
missing cells,0
duplicate line,0
duplicate data,0
missing days calendar,0
erros casual + registered = cnt,0
negative count,0


,count,mean,min,25%,50%,75%,max,std
instant,731.0,366.0,1.0,183.5,366.0,548.5,731.0,211.165812
dteday,731,2012-01-01 00:00:00,2011-01-01 00:00:00,2011-07-02 12:00:00,2012-01-01 00:00:00,2012-07-01 12:00:00,2012-12-31 00:00:00,NaN
season,731.0,2.49658,1.0,2.0,3.0,3.0,4.0,1.110807
yr,731.0,0.500684,0.0,0.0,1.0,1.0,1.0,0.500342
mnth,731.0,6.519836,1.0,4.0,7.0,10.0,12.0,3.451913
holiday,731.0,0.028728,0.0,0.0,0.0,0.0,1.0,0.167155
weekday,731.0,2.997264,0.0,1.0,3.0,5.0,6.0,2.004787
workingday,731.0,0.683995,0.0,0.0,1.0,1.0,1.0,0.465233
weathersit,731.0,1.395349,1.0,1.0,1.0,2.0,3.0,0.544894
temp,731.0,0.495385,0.05913,0.337083,0.498333,0.655417,0.861667,0.183051


Data Preparation: 731 Days and 15 colunms. No lines were removed.


Referência
https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html (Accessed:).